In [82]:
import numpy as np
import pandas as pd

In [83]:
import torch
import matplotlib.pyplot as plt


In [84]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [85]:
df=pd.read_csv('/content/fashion-mnist_train.csv')

In [86]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [87]:
from sklearn.preprocessing import LabelEncoder,OneHotEncoder,StandardScaler
from sklearn.model_selection import train_test_split

In [88]:
X=df.iloc[:,1:].values
y=df.iloc[:,0].values

In [89]:
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2)

In [90]:
x_train_scaled=x_train/255
x_test_scaled=x_test/255

In [91]:
x_train_tensor=torch.from_numpy(x_train_scaled).float()
x_test_tensor=torch.from_numpy(x_test_scaled).float()
y_train_tensor=torch.from_numpy(y_train).long()
y_test_tensor=torch.from_numpy(y_test).long()

In [92]:
from torch.utils.data import Dataset,DataLoader

In [93]:
class Customdataset(Dataset):
  def __init__(self,x,y):
    self.x=x
    self.y=y
  def __len__(self):
    return len(self.x)
  def __getitem__(self, index):
    return self.x[index],self.y[index]

In [94]:
train_dataset=Customdataset(x_train_tensor,y_train_tensor)
test_dataset=Customdataset(x_test_tensor,y_test_tensor)

In [95]:
import torch.nn as nn
from sklearn.metrics import accuracy_score

In [96]:
class NewNeuralNetwork(nn.Module):
  def __init__(self,input_dim,output_dim,dropout_rate,neurons_per_layers):
    super().__init__()
    layers=[]
    for neurons_per_layer in neurons_per_layers:
      layers.append(nn.Linear(input_dim,neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim=neurons_per_layer
    layers.append(nn.Linear(input_dim,output_dim))
    self.model=nn.Sequential(*layers)
  def forward(self,X):
    return self.model(X)


In [97]:
def objective(trial):
  num_hidden_layers=trial.suggest_int('num_hidden_layers',1,5)
  neurons_per_layers=[]
  for i in range(num_hidden_layers):
      layer_size = trial.suggest_int(f"layer_{i}_size", 16,256, step=16)
      neurons_per_layers.append(layer_size)
  learning_rate=trial.suggest_float('learning_rate',1e-5,1e-1,log=True)
  dropout_rate=trial.suggest_float('dropout_rate',0.1,0.5,step=0.1)
  epochs=trial.suggest_int('epochs',10,100,step=10)
  batch_size=trial.suggest_int('batch_size',16,128,step=16)
  optimizer_name=trial.suggest_categorical('optimizer',['adam','sgd','RMSProp'])
  weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

  test_dataloader=DataLoader(test_dataset,batch_size=batch_size,shuffle=True)
  train_dataloader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True)

  input_dim=x_train_tensor.shape[1]
  output_dim=10

  model=NewNeuralNetwork(input_dim,output_dim,dropout_rate,neurons_per_layers)
  model.to(device)
  loss_function=nn.CrossEntropyLoss()
  if optimizer_name=='adam':
    optimizer=torch.optim.Adam(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
  elif optimizer_name=='sgd':
    optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
  elif optimizer_name=='RMSProp':
    optimizer=torch.optim.RMSprop(model.parameters(),lr=learning_rate,weight_decay=weight_decay)
  for i in range(epochs):
    for x_train_batch,y_train_batch in train_dataloader:
      x_train_batch,y_train_batch=x_train_batch.to(device),y_train_batch.to(device)
      y_pred=model(x_train_batch)
      loss=loss_function(y_pred,y_train_batch.long())
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()
  model.eval()
  accuracy_list=[]
  with torch.no_grad():
    for x_test_batch,y_test_batch in test_dataloader:
      x_test_batch,y_test_batch=x_test_batch.to(device),y_test_batch.to(device)
      y_pred=model(x_test_batch)
      y_pred=torch.argmax(y_pred,dim=1)
      accuracy=accuracy_score(y_pred.cpu().numpy(),y_test_batch.cpu().numpy())
      accuracy_list.append(accuracy)
    result=sum(accuracy_list)/len(accuracy_list)
  return result





In [98]:
!pip install optuna

In [99]:
import optuna

study = optuna.create_study(direction='maximize')

[I 2026-08-09 09:12:18,675] A new study created in memory with name: no-name-999e6844-d303-43d5-ac88-232b52b636f0


In [100]:
study.optimize(objective,n_trials=10)

[I 2026-08-09 09:15:32,946] Trial 0 finished with value: 0.40800000000000003 and parameters: {'num_hidden_layers': 4, 'layer_0_size': 64, 'layer_1_size': 80, 'layer_2_size': 16, 'layer_3_size': 80, 'learning_rate': 0.046913926697105056, 'dropout_rate': 0.5, 'epochs': 90, 'batch_size': 80, 'optimizer': 'adam', 'weight_decay': 0.00031940965810289617}. Best is trial 0 with value: 0.40800000000000003.
[I 2026-08-09 09:18:57,802] Trial 1 finished with value: 0.7711666666666667 and parameters: {'num_hidden_layers': 3, 'layer_0_size': 16, 'layer_1_size': 32, 'layer_2_size': 112, 'learning_rate': 0.0044221578685725025, 'dropout_rate': 0.5, 'epochs': 50, 'batch_size': 32, 'optimizer': 'RMSProp', 'weight_decay': 2.2237964490559305e-05}. Best is trial 1 with value: 0.7711666666666667.
[I 2026-08-09 09:19:54,812] Trial 2 finished with value: 0.16040558510638298 and parameters: {'num_hidden_layers': 5, 'layer_0_size': 112, 'layer_1_size': 96, 'layer_2_size': 128, 'layer_3_size': 112, 'layer_4_size'

In [101]:
study

In [103]:
study.best_params

{'num_hidden_layers': 3,
 'layer_0_size': 80,
 'layer_1_size': 48,
 'layer_2_size': 128,
 'learning_rate': 4.91263626381105e-05,
 'dropout_rate': 0.5,
 'epochs': 40,
 'batch_size': 48,
 'optimizer': 'adam',
 'weight_decay': 1.0889708358283822e-05}

In [104]:
study.best_value

0.866